In [ ]:
!pip install git+https://github.com/jacobgil/pytorch-grad-cam.git -q


In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch.cuda.amp as amp  

from torchvision.models import convnext_small   # 🔁 CHANGED MODEL IMPORT
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

import warnings
warnings.filterwarnings("ignore")


In [ ]:
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
original_dirs = {
    'Calculus': '/kaggle/input/oral-diseases/Calculus/Calculus',
    'Caries': '/kaggle/input/oral-diseases/Data caries/Data caries/caries augmented data set/preview',
    'Gingivitis': '/kaggle/input/oral-diseases/Gingivitis/Gingivitis',
    'Ulcers': '/kaggle/input/oral-diseases/Mouth Ulcer/Mouth Ulcer/Mouth_Ulcer_augmented_DataSet/preview',
    'Tooth Discoloration': '/kaggle/input/oral-diseases/Tooth Discoloration/Tooth Discoloration /Tooth_discoloration_augmented_dataser/preview',
    'Hypodontia': '/kaggle/input/oral-diseases/hypodontia/hypodontia'
}


In [ ]:
base_dir = '/kaggle/working/dataset'
splits = ['train', 'val', 'test']
classes = list(original_dirs.keys())


In [ ]:
for split in splits:
    for class_name in classes:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)


In [ ]:
def copy_and_count_images(class_name, image_paths):
    train_paths, test_paths = train_test_split(image_paths, test_size=0.1, random_state=42)
    train_paths, val_paths = train_test_split(train_paths, test_size=0.2, random_state=42)
    split_paths = {'train': train_paths, 'val': val_paths, 'test': test_paths}
    for split, paths in split_paths.items():
        for img_path in paths:
            target_path = os.path.join(base_dir, split, class_name, os.path.basename(img_path))
            shutil.copy(img_path, target_path)


In [ ]:
for class_name, original_dir in original_dirs.items():
    image_paths = [os.path.join(root, file) for root, _, files in os.walk(original_dir) 
                   for file in files if file.endswith(('.jpg', '.jpeg', '.png'))]
    if image_paths:
        copy_and_count_images(class_name, image_paths)

print("Images organized successfully.")


In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [ ]:
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

train_dataset = ImageFolder(train_dir, transform=train_transforms)
val_dataset = ImageFolder(val_dir, transform=val_transforms)
test_dataset = ImageFolder(test_dir, transform=val_transforms)


In [ ]:
batch_size = 32  
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
class CustomConvNeXt(nn.Module):
    def __init__(self, num_classes):
        super(CustomConvNeXt, self).__init__()
        
        self.model = convnext_small(weights="IMAGENET1K_V1")
        
        # Freeze backbone
        for param in self.model.features.parameters():
            param.requires_grad = False
        
        # Replace classifier
        in_features = self.model.classifier[2].in_features
        self.model.classifier[2] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)


In [ ]:
num_classes = len(train_dataset.classes)
model = CustomConvNeXt(num_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=3)


In [ ]:
scaler = amp.GradScaler()


In [ ]:
num_epochs = 50
best_val_loss = float('inf')
patience = 5
epochs_no_improve = 0

train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        with amp.autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            with amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss /= len(val_dataset)
    val_acc = val_correct / val_total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping after {epoch+1} epochs.')
            break
    
    scheduler.step(val_loss)

# Load best model
model.load_state_dict(torch.load('best_model.pth'))

# Test-time augmentation (TTA) with fix
def tta_predict(model, loader, num_augmentations=5):
    model.eval()
    test_preds, test_true = [], []
    tta_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            batch_preds = []
            for _ in range(num_augmentations):
                # Denormalize
                denorm_inputs = inputs * torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1) + \
                                torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
                denorm_inputs = denorm_inputs.clamp(0, 1) * 255
                denorm_inputs = denorm_inputs.byte()
                
                aug_inputs = torch.stack([tta_transforms(Image.fromarray(
                    denorm_inputs[i].cpu().permute(1, 2, 0).numpy())) 
                    for i in range(inputs.size(0))]).to(device)
                outputs = model(aug_inputs)
                batch_preds.append(outputs.softmax(dim=1))
            avg_preds = torch.stack(batch_preds).mean(0)
            _, predicted = torch.max(avg_preds, 1)
            test_preds.extend(predicted.cpu().numpy())
            test_true.extend(labels.cpu().numpy())
    
    return test_true, test_preds

# Evaluate with TTA
test_true, test_preds = tta_predict(model, test_loader)
test_acc = accuracy_score(test_true, test_preds)
print(f'Test Accuracy with TTA: {test_acc:.4f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Validation Loss')
ax1.set_title('Loss Curves')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(train_accuracies, label='Train Accuracy')
ax2.plot(val_accuracies, label='Validation Accuracy')
ax2.set_title('Accuracy Curves')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
class_names = test_dataset.classes

print("Classification Report:")
print(classification_report(test_true, test_preds, target_names=class_names))

conf_mat = confusion_matrix(test_true, test_preds)

plt.figure(figsize=(6, 6))
sns.heatmap(
    conf_mat,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
def visualize_grad_cam(image_paths, model, device, class_names):
    num_images = len(image_paths)
    num_rows = (num_images + 1) // 2
    plt.figure(figsize=(20, 5 * num_rows))
    
    # ✅ Use LAST CONV layer BEFORE classifier
    target_layer = model.model.features[-1][0].block[0]
    grad_cam = GradCAM(model=model, target_layers=[target_layer])
    
    for idx, image_path in enumerate(image_paths):
        input_image = Image.open(image_path).convert('RGB')
        preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
        input_tensor = preprocess(input_image).unsqueeze(0).to(device)
        
        # 🔧 Ensure gradients flow
        input_tensor.requires_grad = True
        
        grayscale_cam = grad_cam(input_tensor)[0]
        input_image_np = np.array(input_image.resize((224, 224))) / 255.0
        visualization = show_cam_on_image(input_image_np, grayscale_cam, use_rgb=True)
        
        outputs = model(input_tensor)
        _, predicted = torch.max(outputs, 1)
        predicted_class = class_names[predicted.item()]
        true_class = os.path.basename(os.path.dirname(image_path))
        title_color = 'green' if true_class == predicted_class else 'red'
        
        plt.subplot(num_rows, 4, 2 * idx + 1)
        plt.imshow(input_image_np)
        plt.title(f'True: {true_class}', fontsize=24, color=title_color)
        plt.axis('off')
        
        plt.subplot(num_rows, 4, 2 * idx + 2)
        plt.imshow(visualization)
        plt.title(f'Predicted: {predicted_class}', fontsize=24, color=title_color)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()


# Get random test images and visualize Grad-CAM
random_images = [
    os.path.join(test_dir, cls, random.choice(os.listdir(os.path.join(test_dir, cls))))
    for cls in classes for _ in range(2)
]

visualize_grad_cam(random_images, model, device, class_names)
